In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
import os

PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')

df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']])
})
print(f"Dữ liệu sẵn sàng! Train: {len(df_train)} câu | Valid: {len(df_valid)} câu")

✅ Dữ liệu sẵn sàng! Train: 54626 câu | Valid: 7310 câu


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128  
    )

print("Đang Tokenize bằng AraBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print("Hoàn tất Tokenize!")

⏳ Đang Tokenize bằng AraBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

✅ Hoàn tất Tokenize!


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F

print("TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...")
# 1. Calculate balanced class weights
y_train = df_train['label'].values
cw = compute_class_weight('balanced', classes=np.arange(19), y=y_train)

# 2. Clip class weights
cw_clipped = np.clip(cw, 0.5, 5.0)

# 3. Normalize class weights
cw_normalized = cw_clipped / cw_clipped.mean()
class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

print("Class weights (đã clip và chuẩn hóa):")
print(np.round(cw_normalized, 2))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=18
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)

class CORALTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        device = logits.device
        
        target = torch.zeros_like(logits, device=device)
        for k in range(logits.shape[1]):
            target[:, k] = (labels > k).float()
            
        loss_per_sample = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none'
        ).mean(dim=1)
        
        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels] 
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_coral(eval_pred):
    logits, labels = eval_pred
    preds_binary = logits > 0
    pred_labels = preds_binary.sum(axis=1)
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    return {"qwk": qwk}

⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...
🔸 Class weights (đã clip và chuẩn hóa):
[2.02 2.02 1.02 1.99 0.44 0.97 0.28 0.26 0.73 0.2  0.29 0.2  0.36 0.2
 0.58 1.35 2.02 2.02 2.02]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="../saved_models/arabert_lora_coral_sample_weight",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,   
    fp16=False,  
    seed=42
)

trainer = CORALTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_coral,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights_tensor  
)

print("BẮT ĐẦU HUẤN LUYỆN (VỚI BFLOAT16 + CLIPPED SAMPLE WEIGHTS)...")
trainer.train()

trainer.save_model("../saved_models/arabert_lora_coral_best")
tokenizer.save_pretrained("../saved_models/arabert_lora_coral_best")

🚀 BẮT ĐẦU HUẤN LUYỆN (VỚI BFLOAT16 + CLIPPED SAMPLE WEIGHTS)...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 0.0883, 'grad_norm': 3.767958879470825, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 0.0724, 'grad_norm': 0.9471854567527771, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 0.0683, 'grad_norm': 0.9950404763221741, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06881075352430344, 'eval_qwk': 0.7946676806459272, 'eval_runtime': 30.3929, 'eval_samples_per_second': 240.517, 'eval_steps_per_second': 15.036, 'epoch': 1.0}
{'loss': 0.0614, 'grad_norm': 1.262100338935852, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 0.0563, 'grad_norm': 1.834261178970337, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 0.057, 'grad_norm': 0.7713063359260559, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06585898250341415, 'eval_qwk': 0.8195043402299107, 'eval_runtime': 30.4067, 'eval_samples_per_second': 240.408, 'eval_steps_per_second': 15.03, 'epoch': 4.0}
{'loss': 0.0398, 'grad_norm': 0.3483574092388153, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.0371, 'grad_norm': 0.5743739008903503, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.0372, 'grad_norm': 0.7595680356025696, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.0355, 'grad_norm': 0.8126291036605835, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06984170526266098, 'eval_qwk': 0.8152997760605669, 'eval_runtime': 30.5578, 'eval_samples_per_second': 239.218, 'eval_steps_per_second': 14.955, 'epoch': 5.0}
{'train_runtime': 4944.0188, 'train_samples_per_second': 55.245, 'train_steps_per_second': 1.726, 'train_loss': 0.05154999194958088, 'epoch': 5.0}


('../saved_models/arabert_lora_coral_best\\tokenizer_config.json',
 '../saved_models/arabert_lora_coral_best\\special_tokens_map.json',
 '../saved_models/arabert_lora_coral_best\\vocab.txt',
 '../saved_models/arabert_lora_coral_best\\added_tokens.json',
 '../saved_models/arabert_lora_coral_best\\tokenizer.json')

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score, classification_report

print("ĐANG LẤY LOGITS VÀ NHÃN TỪ TẬP VALIDATION...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions  
true_labels = predictions_output.label_ids.astype(int)

print("ĐANG DÒ TÌM GLOBAL SHIFT (DELTA) TỐI ƯU...")

best_delta = 0.0
best_qwk = -1.0
qwk_history = []
deltas = np.arange(-2.0, 2.0, 0.01) 

for delta in deltas:
    preds_binary = logits > delta
    pred_labels = preds_binary.sum(axis=1)
    
    qwk = cohen_kappa_score(true_labels, pred_labels, weights='quadratic')
    
    if qwk > best_qwk:
        best_qwk = qwk
        best_delta = delta

print(f"Điểm dịch chuyển (Delta) tối ưu : {best_delta:.2f}")
print(f"QWK SAU KHI DỊCH CHUYỂN         : {best_qwk:.4f}")

final_preds_binary = logits > best_delta
final_pred_labels = final_preds_binary.sum(axis=1)

print("=== BÁO CÁO F1-SCORE VỚI DELTA TỐI ƯU ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

🔍 ĐANG DÒ TÌM GLOBAL SHIFT (DELTA) TỐI ƯU...
🎯 Điểm dịch chuyển (Delta) tối ưu : -0.37
🚀 QWK SAU KHI DỊCH CHUYỂN         : 0.8242

📊 === BÁO CÁO F1-SCORE VỚI DELTA TỐI ƯU ===
              precision    recall  f1-score   support

     Level_1       0.76      0.73      0.74        44
     Level_2       0.49      0.37      0.42        68
     Level_3       0.50      0.57      0.53       182
     Level_4       0.22      0.69      0.33        78
     Level_5       0.56      0.47      0.51       417
     Level_6       0.32      0.52      0.40       189
     Level_7       0.56      0.52      0.54       701
     Level_8       0.64      0.51      0.57       613
     Level_9       0.36      0.64      0.46       236
    Level_10       0.69      0.65      0.67      1012
    Level_11       0.22      0.25      0.24       409
    Level_12       0.47      0.33      0.39      1491
    Level_13       0.22      0.43      0.30       349
    Level_14       0.56      0.42      0.48      1072
    Level_15  